In [1]:
pip install dash plotly pandas


   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   ---------------------------------------- 0.1/7.5 MB 1.7 MB/s eta 0:00:05
    --------------------------------------- 0.1/7.5 MB 2.1 MB/s eta 0:00:04
   - -------------------------------------- 0.2/7.5 MB 2.1 MB/s eta 0:00:04
   - -------------------------------------- 0.3/7.5 MB 2.2 MB/s eta 0:00:04
   -- ------------------------------------- 0.4/7.5 MB 2.2 MB/s eta 0:00:04
   -- ------------------------------------- 0.5/7.5 MB 2.2 MB/s eta 0:00:04
   --- ------------------------------------ 0.6/7.5 MB 2.4 MB/s eta 0:00:03
   ---- ----------------------------------- 0.8/7.5 MB 2.4 MB/s eta 0:00:03
   ---- ----------------------------------- 0.9/7.5 MB 2.5 MB/s eta 0:00:03
   ----- ---------------------------------- 1.0/7.5 MB 2.6 MB/s eta 0:00:03
   ------ --------------------------------- 1.2/7.5 MB 2.6 MB/s eta 0:00:03
   ------- -------------------------------- 1.4/7.5 MB 2.4 MB/s eta 0:00:03
   -------- -------


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
from dash import Dash, dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

# Load your dataset
dfs = pd.read_csv('historical_automobile_sales.csv')
dfs['Month'] = pd.to_datetime(df['Date']).dt.month_name()
dfs['Year'] = pd.to_datetime(df['Date']).dt.year

# Create Dash app
app = Dash(__name__)

# Layout for the Dash app
app.layout = html.Div([
    html.H1('Automobile Sales Dashboard'),

    html.Label('Type:'),
    dcc.Dropdown(options=[{'label': 'Recession', 'value': 'Recession'},
                          {'label': 'Non-Recession', 'value': 'Non-Recession'}],
                 value='Recession', id='period_type'),

    html.Label('Select Year:'),
    dcc.Dropdown(options=[{'label': str(year), 'value': year} for year in df['Year'].unique()],
                 value=2005, id='year'),

    html.Div([dcc.Graph(id='plot1'), dcc.Graph(id='plot2')])
])

@app.callback(
    [Output('plot1', 'figure'), Output('plot2', 'figure')],
    [Input('period_type', 'value'), Input('year', 'value')]
)
def update_graphs(type, selected_year):
    recession_years = [1980, 1981, 1982, 1991, 2000, 2001, 2007, 2008, 2009, 2020]
    
    if type == 'Recession':
        filtered_data = dfs[dfs['Year'].isin(recession_years) & (dfs['Year'] == selected_year)]
    else:
        filtered_data = dfs[~dfs['Year'].isin(recession_years) & (dfs['Year'] == selected_year)]

    # Pie chart for sales distribution by vehicle type
    sales_data = filtered_data.groupby('Vehicle_Type')['Automobile_Sales'].sum().reset_index()
    plt_1 = px.pie(sales_data, values='Automobile_Sales', names='Vehicle_Type', 
                  title=f'Sales Distribution ({type} in {selected_year})')

    # Bar chart for monthly sales
    monthly_sales = filtered_data.groupby('Month')['Automobile_Sales'].sum().reset_index()
    plt_2 = px.bar(monthly_sales, x='Month', y='Automobile_Sales',
                  title=f'Monthly Sales ({selected_year}, {type})')

    return plt_1, plt_2

# Run the app
if __name__ == '__main__':
    app.run_server()
